# LoRA Fine-Tuning -- Low-Rank Adaptation

## Why LoRA?

Full fine-tuning of a large language model updates **every single parameter**. For a 7-billion-parameter model, that means storing optimizer states for all 7B weights, which can easily require **60+ GB of VRAM** -- far beyond what consumer GPUs offer.

**LoRA (Low-Rank Adaptation)** solves this by freezing the original model weights and injecting small, trainable matrices into selected layers. The result:

| Approach | Trainable Params (7B model) | GPU VRAM Required |
|---|---|---|
| Full Fine-Tuning | ~7,000,000,000 | 60+ GB |
| LoRA (rank=16) | ~5,000,000 | 8-16 GB |
| **Reduction** | **~100x fewer** | **~4-8x less** |

This means you can fine-tune powerful models on a **single consumer GPU** (RTX 3090, RTX 4090) or even for free on Google Colab.

## What You Will Learn

1. **The math** behind low-rank adaptation (W_new = W + BA)
2. **How low-rank decomposition** captures weight updates efficiently
3. **How to configure LoRA** -- rank, alpha, target modules, and dropout
4. **Training tips** and common pitfalls to avoid

## Prerequisites

- Basic understanding of neural networks and matrix multiplication
- Familiarity with PyTorch and Hugging Face Transformers
- Read the source implementation: `src/agentexplorr/llm_training/fine_tune_lora.py`

## Learning Resources

- **Paper**: [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) (Hu et al., 2021)
- **PEFT Library**: [HuggingFace PEFT docs](https://huggingface.co/docs/peft)
- **Video**: [LoRA Explained in 10 Minutes](https://www.youtube.com/watch?v=PXWYUTMt-AU) (Umar Jamil)
- **Blog**: [Practical Tips for Finetuning LLMs Using LoRA](https://magazine.sebastianraschka.com/p/practical-tips-for-finetuning-llms) (Sebastian Raschka)

In [ ]:
import numpy as np

# ──────────────────────────────────────────────────────────────────────
# The Core LoRA Equation
# ──────────────────────────────────────────────────────────────────────
# In a standard linear layer:
#     y = W @ x            where W has shape (d_out, d_in)
#
# With LoRA, we ADD a low-rank update:
#     y = W @ x + B @ A @ x
#
# Where:
#     W = original frozen weights   (d_out x d_in)  -- NOT updated
#     A = down-projection matrix    (r x d_in)      -- trainable
#     B = up-projection matrix      (d_out x r)     -- trainable
#     r = rank (much smaller than d_in and d_out)
#
# The effective new weight matrix is:
#     W_new = W + B @ A
#
# This works because B @ A has the same shape as W, but is parameterized
# by only r * (d_in + d_out) values instead of d_in * d_out.
# ──────────────────────────────────────────────────────────────────────

# Let's see the parameter savings concretely
d = 4096   # Typical hidden dimension for a 7B model
r = 16     # LoRA rank

full_params = d * d
lora_params = r * d * 2   # A is (r x d) and B is (d x r)

print("=" * 55)
print("LoRA Parameter Comparison")
print("=" * 55)
print(f"Hidden dimension (d):       {d}")
print(f"LoRA rank (r):              {r}")
print(f"Full weight matrix params:  {full_params:>12,}")
print(f"LoRA A + B params:          {lora_params:>12,}")
print(f"Reduction factor:           {full_params / lora_params:>12.0f}x")
print(f"LoRA is {100 * lora_params / full_params:.2f}% of full params")

## The Math of Low-Rank Adaptation

### Why "Low-Rank"?

The key insight from the LoRA paper is that **weight updates during fine-tuning have low intrinsic rank**. In other words, the "information" learned during fine-tuning lives in a much smaller subspace than the full weight matrix.

Think of it like image compression: a 4K photo can be compressed to 1/10th its size because most of the pixel information is redundant. LoRA does the same thing for neural network weight updates.

### The Decomposition

A standard weight update `delta_W` has shape `(d x d)`. LoRA decomposes it into two smaller matrices:

```
delta_W  =  B    @    A
(d x d)    (d x r)   (r x d)
```

Where `r << d` (e.g., r=16 when d=4096).

### Visual Diagram

```
Standard Fine-Tuning:                 LoRA Fine-Tuning:

  x ──> [ W + delta_W ] ──> y          x ──> [ W (frozen) ] ──────> (+) ──> y
                                        |                             ^
        d x d parameters                └──> [ A ] ──> [ B ] ────────┘
        (all trainable)                      (r x d)   (d x r)
                                             only r*(d+d) trainable
```

### Scaling Factor

LoRA applies a scaling factor `alpha / r` to the low-rank update:

```
y = W @ x + (alpha / r) * B @ A @ x
```

- **alpha = r** gives a scaling of 1.0 (conservative)
- **alpha = 2*r** gives a scaling of 2.0 (slightly aggressive, commonly used)
- Higher alpha amplifies the LoRA update, making the model change more during training

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Demo: Low-Rank Decomposition with NumPy
# ──────────────────────────────────────────────────────────────────────
# Let's see how a low-rank approximation captures most of the information
# in a matrix, using Singular Value Decomposition (SVD).

np.random.seed(42)

# Simulate a weight update matrix (d x d) that has low intrinsic rank.
# In practice, the LoRA paper found that fine-tuning updates are like this:
# most of the "energy" is concentrated in the top few singular values.
d = 256  # Using smaller dimension for demo speed

# Create a matrix that is APPROXIMATELY low-rank (like real weight updates)
# We generate it as a sum of a low-rank component + small noise
true_rank = 8
low_rank_component = np.random.randn(d, true_rank) @ np.random.randn(true_rank, d)
noise = 0.1 * np.random.randn(d, d)
delta_W = low_rank_component + noise

print(f"Original delta_W shape: {delta_W.shape}")
print(f"Original delta_W params: {d * d:,}")
print()

# Now approximate delta_W using different ranks via SVD
U, S, Vt = np.linalg.svd(delta_W, full_matrices=False)

print(f"{'Rank':>6} | {'LoRA Params':>12} | {'Reconstruction Error':>20} | {'% of Original':>14}")
print("-" * 65)

for rank in [1, 2, 4, 8, 16, 32, 64]:
    # Low-rank approximation: keep only top-`rank` singular values
    B = U[:, :rank] * S[:rank]   # shape: (d, rank) -- like LoRA's B matrix
    A = Vt[:rank, :]             # shape: (rank, d) -- like LoRA's A matrix

    # Reconstruct the approximation
    approx = B @ A

    # Measure reconstruction quality
    error = np.linalg.norm(delta_W - approx, 'fro')
    original_norm = np.linalg.norm(delta_W, 'fro')
    relative_error = error / original_norm * 100
    lora_params = rank * d * 2

    print(f"{rank:>6} | {lora_params:>12,} | {relative_error:>19.2f}% | {100 * lora_params / (d*d):>13.1f}%")

print()
print("Notice: rank=8 captures almost all the information (very low error)")
print("because our simulated weight update has true rank 8.")
print("Real fine-tuning updates behave similarly -- rank 16-64 is usually enough.")

## Configuring LoRA: Key Parameters

When setting up LoRA for fine-tuning, you need to choose several hyperparameters. Here is a reference guide:

### Rank (`r`)
The most important LoRA hyperparameter. Controls how many parameters are added.

| Rank | Use Case | Trainable Params (per layer) |
|------|----------|------------------------------|
| 8 | Simple tasks (classification, sentiment) | Minimal |
| 16 | **Good default** for instruction following | Moderate |
| 32 | Complex tasks, larger datasets | Higher |
| 64 | Approaching full fine-tuning quality | Maximum common |

### Alpha (`lora_alpha`)
Scaling factor applied as `alpha / r`. Controls how strongly LoRA modifies the output.
- **Common practice**: Set `alpha = 2 * r` (e.g., r=16, alpha=32)
- If training is unstable (loss spikes), reduce alpha
- If the model is not learning enough, increase alpha

### Target Modules (`target_modules`)
Which layers in the Transformer to apply LoRA to:

- **Attention layers**: `q_proj`, `k_proj`, `v_proj`, `o_proj`
- **MLP layers**: `gate_proj`, `up_proj`, `down_proj`
- The original LoRA paper only targeted `q_proj` and `v_proj`
- Later research (QLoRA) showed that targeting **all linear layers** often gives better results

### Dropout (`lora_dropout`)
Regularization to prevent overfitting:
- `0.0` -- No dropout, fine for large datasets (>10K examples)
- `0.05` -- Light dropout, good default
- `0.1` -- Moderate, good for small datasets (<1K examples)

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Configuring LoRA in Practice with PEFT
# ──────────────────────────────────────────────────────────────────────
# Below is the standard way to set up LoRA using the Hugging Face PEFT
# library. This mirrors the configuration in our source module:
#   src/agentexplorr/llm_training/fine_tune_lora.py

# NOTE: This cell shows the configuration pattern. Running it requires
# the peft and transformers libraries installed with a GPU available.
# We print the config dict instead of instantiating to keep this
# notebook runnable on any machine.

# --- LoRA Configuration ---
lora_config = {
    "task_type": "CAUSAL_LM",        # Decoder-only model (GPT/Llama style)
    "r": 16,                          # Rank -- start here, try 8 or 32
    "lora_alpha": 32,                 # Scaling factor -- alpha/r = 2.0
    "lora_dropout": 0.05,             # Light dropout for regularization
    "target_modules": [
        "q_proj",                     # Query projection (self-attention)
        "k_proj",                     # Key projection (self-attention)
        "v_proj",                     # Value projection (self-attention)
        "o_proj",                     # Output projection (self-attention)
        "gate_proj",                  # Gate in MLP (Llama-style models)
        "up_proj",                    # Up projection in MLP
        "down_proj",                  # Down projection in MLP
    ],
    "bias": "none",                   # Don't train bias terms (minimal benefit)
}

# --- Training Configuration ---
training_config = {
    "num_train_epochs": 3,            # 3 passes through the data
    "per_device_train_batch_size": 4,  # Examples per GPU per step
    "gradient_accumulation_steps": 4,  # Effective batch = 4*4 = 16
    "learning_rate": 2e-4,            # Sweet spot for LoRA (1e-4 to 3e-4)
    "warmup_ratio": 0.03,            # Warm up LR for first 3% of steps
    "weight_decay": 0.01,            # Light L2 regularization
    "lr_scheduler_type": "cosine",    # Cosine decay -- standard for LLMs
    "max_grad_norm": 1.0,            # Gradient clipping for stability
    "fp16": False,                    # Set True for NVIDIA GPUs
    "bf16": False,                    # Set True for Ampere+ GPUs (RTX 3000+)
}

# Display the configurations
print("=" * 55)
print("LoRA Configuration")
print("=" * 55)
for key, value in lora_config.items():
    if isinstance(value, list):
        print(f"  {key}:")
        for item in value:
            print(f"    - {item}")
    else:
        print(f"  {key}: {value}")

print()
print("=" * 55)
print("Training Configuration")
print("=" * 55)
for key, value in training_config.items():
    print(f"  {key}: {value}")

# Calculate effective batch size
eff_batch = training_config["per_device_train_batch_size"] * training_config["gradient_accumulation_steps"]
print(f"\n  --> Effective batch size: {eff_batch}")
print(f"  --> LoRA scaling factor (alpha/r): {lora_config['lora_alpha'] / lora_config['r']}")

# ──────────────────────────────────────────────────────────────────────
# To actually create these objects with PEFT, you would do:
#
#   from peft import LoraConfig, TaskType, get_peft_model
#   from transformers import AutoModelForCausalLM
#
#   peft_config = LoraConfig(**lora_config)
#   model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
#   model = get_peft_model(model, peft_config)
#   model.print_trainable_parameters()
#   # Output: "trainable params: 4,194,304 || all params: 1,100,048,384 || trainable%: 0.38"
# ──────────────────────────────────────────────────────────────────────

## Training Tips and Common Pitfalls

### Getting Started Checklist

1. **Start with TinyLlama** (1.1B params) for fast iteration, then scale to larger models
2. **Use `r=16, alpha=32`** as your starting point
3. **Target all linear layers** for best quality (not just q_proj/v_proj)
4. **Set learning rate to 2e-4** and tune from there (try 1e-4 and 5e-4)

### Common Pitfalls

**CUDA Out of Memory (OOM)**
- Reduce `per_device_train_batch_size` to 2 or 1
- Increase `gradient_accumulation_steps` to compensate
- Enable gradient checkpointing (`gradient_checkpointing=True`)
- Use mixed precision: `bf16=True` on Ampere+ GPUs, `fp16=True` otherwise

**Loss is Not Decreasing**
- Learning rate too low -- try increasing to 5e-4
- LoRA rank too small -- try increasing to 32 or 64
- Alpha too low -- try `alpha = 2 * r` or even `4 * r`
- Data quality issue -- inspect your training examples

**Loss Goes to NaN or Spikes**
- Learning rate too high -- try reducing to 1e-4 or 5e-5
- Alpha too high -- reduce to match `r` (scaling factor = 1.0)
- Try enabling gradient clipping: `max_grad_norm=1.0`
- Switch from `fp16` to `bf16` (more numerically stable)

**Model Overfits (train loss low, val loss high)**
- Increase `lora_dropout` (try 0.1)
- Reduce `num_train_epochs` (try 1-2)
- Increase `weight_decay` (try 0.05 or 0.1)
- Get more training data or use data augmentation

### Saving and Loading

- **Adapter-only save** (~10-50 MB): Just the LoRA weights. Load with `PeftModel.from_pretrained(base_model, adapter_path)`.
- **Merged save** (~2-14 GB): LoRA weights baked into the base model. Simpler inference, but larger file and no adapter swapping.

## Key Takeaways

1. **LoRA freezes the original model** and adds small trainable matrices (`B @ A`) to selected layers, reducing trainable parameters by ~100x.

2. **The math is simple**: `W_new = W + B @ A`, where `B` is `(d x r)` and `A` is `(r x d)`. The rank `r` is typically 8-64, far smaller than `d` (often 4096+).

3. **Low-rank works** because fine-tuning weight updates naturally have low intrinsic dimensionality -- most of the "information" lives in a small subspace.

4. **Key hyperparameters to tune** (in order of importance):
   - `learning_rate`: Start at 2e-4
   - `num_train_epochs`: Start at 3
   - `lora_r`: Start at 16
   - `per_device_train_batch_size`: As high as VRAM allows

5. **LoRA adapters are tiny** (~10-50 MB) and can be swapped, shared, and stacked -- enabling multi-task deployment from a single base model.

## Next Steps

- **Run the full pipeline**: See `src/agentexplorr/llm_training/fine_tune_lora.py` for the complete `LoRAFineTuner` class
- **Evaluate your model**: Continue to `03_evaluation.ipynb` to learn how to measure model quality
- **Try QLoRA**: Combine LoRA with 4-bit quantization for even lower memory usage (see the QLoRA source module)
- **Explore the PEFT library**: [HuggingFace PEFT docs](https://huggingface.co/docs/peft) cover additional methods like AdaLoRA, IA3, and prefix tuning